# Model Evaluation and Diagnostics\n\n**Chapter 6: Regression Techniques for Soccer Analytics**\n\n## Learning Objectives\n\n- Master key regression performance metrics (R², RMSE, MAE)\n- Understand the importance of train/test splits and validation sets\n- Perform residual analysis to diagnose model problems\n- Identify and handle influential observations\n- Compare multiple models systematically\n- Interpret model coefficients in soccer context

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.model_selection import train_test_split, cross_val_score\nfrom sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error\nimport statsmodels.api as sm\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (12, 8)

## Key Performance Metrics\n\nFor regression models, we have several metrics to evaluate performance:\n\n| Metric | Formula | Interpretation | Range |\n|--------|---------|----------------|-------|\n| **R²** | 1 - (SS_res / SS_tot) | % of variance explained | 0 to 1 (higher better) |\n| **RMSE** | √(mean(errors²)) | Average prediction error (same units) | 0 to ∞ (lower better) |\n| **MAE** | mean(\|errors\|) | Average absolute error | 0 to ∞ (lower better) |\n| **MAPE** | mean(\|errors/actual\|) × 100 | Average % error | 0 to ∞ (lower better) |

## Load Data: Match Outcomes\n\nWe'll use match data to predict goals scored.

In [ ]:
# Simulated match data\nnp.random.seed(42)\nn_matches = 100\n\ndata = {\n    'ShotsOnTarget': np.random.randint(3, 15, n_matches),\n    'Possession': np.random.uniform(35, 65, n_matches),\n    'xG': np.random.uniform(0.5, 3.5, n_matches),\n    'Goals': np.random.randint(0, 5, n_matches)\n}\n\ndf = pd.DataFrame(data)\n# Add realistic correlation\ndf['Goals'] = (df['xG'] * 0.8 + df['ShotsOnTarget'] * 0.1 + \n               np.random.normal(0, 0.5, n_matches)).clip(0, 5).round().astype(int)\n\nprint(\"Match Dataset:\")\nprint(df.head(10))\nprint(f\"\\nDataset shape: {df.shape}\")\nprint(f\"\\nBasic statistics:\")\nprint(df.describe())

## Train/Test Split: The Foundation of Evaluation\n\n**Critical concept:** Never evaluate on training data!\n\n- **Training set:** Used to fit the model\n- **Test set:** Used to evaluate generalization\n- **Validation set:** Used for hyperparameter tuning (optional)\n\n**Why?** Training performance is optimistic. Test performance shows real-world capability.

In [ ]:
# Prepare features and target\nfeatures = ['ShotsOnTarget', 'Possession', 'xG']\ntarget = 'Goals'\n\nX = df[features]\ny = df[target]\n\n# Split: 70% train, 30% test\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.3, random_state=42\n)\n\nprint(f\"Training set: {len(X_train)} matches ({len(X_train)/len(df)*100:.0f}%)\")\nprint(f\"Test set: {len(X_test)} matches ({len(X_test)/len(df)*100:.0f}%)\")\nprint(f\"\\nThis ensures unbiased evaluation on unseen data.\")

## Build and Evaluate a Model

In [ ]:
# Train model\nmodel = LinearRegression()\nmodel.fit(X_train, y_train)\n\n# Make predictions\ny_train_pred = model.predict(X_train)\ny_test_pred = model.predict(X_test)\n\n# Calculate metrics\ntrain_r2 = r2_score(y_train, y_train_pred)\ntest_r2 = r2_score(y_test, y_test_pred)\n\ntrain_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))\ntest_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))\n\ntrain_mae = mean_absolute_error(y_train, y_train_pred)\ntest_mae = mean_absolute_error(y_test, y_test_pred)\n\nprint(\"Model Performance:\")\nprint(f\"\\n{'Metric':<15} {'Training':<15} {'Test':<15}\")\nprint(\"-\" * 45)\nprint(f\"{'R²':<15} {train_r2:<15.3f} {test_r2:<15.3f}\")\nprint(f\"{'RMSE':<15} {train_rmse:<15.3f} {test_rmse:<15.3f}\")\nprint(f\"{'MAE':<15} {train_mae:<15.3f} {test_mae:<15.3f}\")\n\nif abs(train_r2 - test_r2) > 0.1:\n    print(\"\\n⚠️ Warning: Large gap between train and test performance suggests overfitting!\")\nelse:\n    print(\"\\n✅ Train and test performance are similar - good generalization!\")

## Interpreting Metrics\n\n**R² = 0.85** means:\n- 85% of variance in goals is explained by our features\n- 15% is unexplained (random variation, missing features)\n\n**RMSE = 0.6 goals** means:\n- On average, predictions are off by 0.6 goals\n- In soccer context: predicting 2.3 goals when actual is 2 or 3\n\n**MAE = 0.5 goals** means:\n- Average absolute error is half a goal\n- More interpretable than RMSE for stakeholders

## Residual Analysis: Diagnosing Problems\n\n**Residuals** = Actual - Predicted\n\nA good model should have:\n1. **Random scatter** around zero (no patterns)\n2. **Constant variance** (homoscedasticity)\n3. **Normal distribution** (for statistical inference)

In [ ]:
# Calculate residuals\nresiduals = y_test - y_test_pred\n\n# Create residual plots\nfig, axes = plt.subplots(2, 2, figsize=(14, 10))\n\n# 1. Residuals vs Predicted\naxes[0, 0].scatter(y_test_pred, residuals, alpha=0.6)\naxes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)\naxes[0, 0].set_xlabel('Predicted Goals')\naxes[0, 0].set_ylabel('Residuals')\naxes[0, 0].set_title('Residuals vs. Predicted Values')\naxes[0, 0].grid(True, alpha=0.3)\n\n# 2. Residuals vs Actual\naxes[0, 1].scatter(y_test, residuals, alpha=0.6)\naxes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)\naxes[0, 1].set_xlabel('Actual Goals')\naxes[0, 1].set_ylabel('Residuals')\naxes[0, 1].set_title('Residuals vs. Actual Values')\naxes[0, 1].grid(True, alpha=0.3)\n\n# 3. Histogram of residuals\naxes[1, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)\naxes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2)\naxes[1, 0].set_xlabel('Residuals')\naxes[1, 0].set_ylabel('Frequency')\naxes[1, 0].set_title('Distribution of Residuals')\n\n# 4. Q-Q plot\nfrom scipy import stats\nstats.probplot(residuals, dist=\"norm\", plot=axes[1, 1])\naxes[1, 1].set_title('Q-Q Plot')\n\nplt.tight_layout()\nplt.show()\n\nprint(\"Residual Analysis Checklist:\")\nprint(\"✓ Random scatter around zero? (No U-shape or pattern)\")\nprint(\"✓ Constant variance? (No funnel shape)\")\nprint(\"✓ Approximately normal? (Histogram bell-shaped, Q-Q plot linear)\")

## Identifying Influential Observations\n\n**Influential points** can disproportionately affect the model. Let's find them!

In [ ]:
# Use statsmodels for influence diagnostics\nX_train_sm = sm.add_constant(X_train)\nmodel_sm = sm.OLS(y_train, X_train_sm).fit()\n\n# Get influence measures\ninfluence = model_sm.get_influence()\nleverage = influence.hat_matrix_diag\ncooks_d = influence.cooks_distance[0]\n\n# Plot Cook's distance\nplt.figure(figsize=(12, 6))\nplt.stem(range(len(cooks_d)), cooks_d, markerfmt=',')\nplt.axhline(y=4/len(X_train), color='r', linestyle='--', label='Threshold (4/n)')\nplt.xlabel('Observation Index')\nplt.ylabel(\"Cook's Distance\")\nplt.title(\"Influential Observations (Cook's Distance)\")\nplt.legend()\nplt.tight_layout()\nplt.show()\n\n# Identify influential points\nthreshold = 4 / len(X_train)\ninfluential_idx = np.where(cooks_d > threshold)[0]\n\nif len(influential_idx) > 0:\n    print(f\"Found {len(influential_idx)} influential observations:\")\n    print(X_train.iloc[influential_idx])\n    print(\"\\nThese points have high leverage on the model. Consider investigating them!\")\nelse:\n    print(\"No highly influential observations detected.\")

## Cross-Validation: Robust Evaluation\n\n**Problem:** Single train/test split can be lucky or unlucky.\n\n**Solution:** K-Fold Cross-Validation\n- Split data into K folds\n- Train on K-1 folds, test on 1\n- Repeat K times\n- Average the results

In [ ]:
# Perform 5-fold cross-validation\nfrom sklearn.model_selection import cross_validate\n\nmodel_cv = LinearRegression()\n\n# Get multiple metrics\nscoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']\ncv_results = cross_validate(model_cv, X, y, cv=5, scoring=scoring)\n\n# Extract and display results\nr2_scores = cv_results['test_r2']\nrmse_scores = np.sqrt(-cv_results['test_neg_mean_squared_error'])\nmae_scores = -cv_results['test_neg_mean_absolute_error']\n\nprint(\"5-Fold Cross-Validation Results:\")\nprint(f\"\\nR² scores: {r2_scores}\")\nprint(f\"  Mean: {r2_scores.mean():.3f} (±{r2_scores.std():.3f})\")\nprint(f\"\\nRMSE scores: {rmse_scores}\")\nprint(f\"  Mean: {rmse_scores.mean():.3f} (±{rmse_scores.std():.3f})\")\nprint(f\"\\nMAE scores: {mae_scores}\")\nprint(f\"  Mean: {mae_scores.mean():.3f} (±{mae_scores.std():.3f})\")\nprint(f\"\\nCross-validation provides more reliable performance estimates!\")

## Model Comparison Framework\n\nLet's compare multiple models systematically.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.preprocessing import StandardScaler\n\n# Define models\nmodels = {\n    'Linear Regression': LinearRegression(),\n    'KNN (K=5)': KNeighborsRegressor(n_neighbors=5),\n    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42)\n}\n\n# Scale features for KNN\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\n\n# Compare models\nresults = []\n\nfor name, model in models.items():\n    # Use scaled data for KNN, original for others\n    X_use = X_scaled if 'KNN' in name else X\n    \n    # Cross-validation\n    cv_scores = cross_val_score(model, X_use, y, cv=5, scoring='r2')\n    \n    # Train on full training set and evaluate on test\n    X_train_use = scaler.transform(X_train) if 'KNN' in name else X_train\n    X_test_use = scaler.transform(X_test) if 'KNN' in name else X_test\n    \n    model.fit(X_train_use, y_train)\n    test_r2 = model.score(X_test_use, y_test)\n    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test_use)))\n    \n    results.append({\n        'Model': name,\n        'CV R² (mean)': cv_scores.mean(),\n        'CV R² (std)': cv_scores.std(),\n        'Test R²': test_r2,\n        'Test RMSE': test_rmse\n    })\n\n# Display comparison\nresults_df = pd.DataFrame(results)\nprint(\"Model Comparison:\")\nprint(results_df.to_string(index=False))\nprint(f\"\\nBest model by CV R²: {results_df.loc[results_df['CV R² (mean)'].idxmax(), 'Model']}\")\nprint(f\"Best model by Test R²: {results_df.loc[results_df['Test R²'].idxmax(), 'Model']}\")

## Visualize Model Comparison

In [ ]:
# Bar plot comparison\nfig, axes = plt.subplots(1, 2, figsize=(14, 6))\n\n# R² comparison\naxes[0].bar(results_df['Model'], results_df['Test R²'], alpha=0.7)\naxes[0].set_ylabel('R² Score')\naxes[0].set_title('Model Comparison: R² Score')\naxes[0].tick_params(axis='x', rotation=45)\naxes[0].grid(True, alpha=0.3)\n\n# RMSE comparison\naxes[1].bar(results_df['Model'], results_df['Test RMSE'], alpha=0.7, color='coral')\naxes[1].set_ylabel('RMSE')\naxes[1].set_title('Model Comparison: RMSE')\naxes[1].tick_params(axis='x', rotation=45)\naxes[1].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## Interpreting Coefficients\n\nFor linear models, coefficients tell us feature importance.

In [ ]:
# Get coefficients from linear model\nlinear_model = LinearRegression()\nlinear_model.fit(X_train, y_train)\n\ncoef_df = pd.DataFrame({\n    'Feature': features,\n    'Coefficient': linear_model.coef_\n}).sort_values('Coefficient', ascending=False)\n\nprint(\"Feature Coefficients:\")\nprint(coef_df.to_string(index=False))\nprint(f\"\\nIntercept: {linear_model.intercept_:.3f}\")\nprint(f\"\\nInterpretation:\")\nfor _, row in coef_df.iterrows():\n    print(f\"  {row['Feature']}: +1 unit → {row['Coefficient']:+.3f} goals\")\n\n# Visualize\nplt.figure(figsize=(10, 6))\nplt.barh(coef_df['Feature'], coef_df['Coefficient'])\nplt.xlabel('Coefficient Value')\nplt.title('Feature Importance (Linear Regression Coefficients)')\nplt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)\nplt.tight_layout()\nplt.show()

## Summary\n\nIn this notebook, we:\n\n1. ✅ Mastered key regression metrics (R², RMSE, MAE)\n2. ✅ Implemented proper train/test splits\n3. ✅ Performed comprehensive residual analysis\n4. ✅ Identified influential observations\n5. ✅ Used cross-validation for robust evaluation\n6. ✅ Compared multiple models systematically\n7. ✅ Interpreted model coefficients\n\n## Key Takeaways\n\n- **Never evaluate on training data** - always use test set or cross-validation\n- **Multiple metrics** give a complete picture (R², RMSE, MAE)\n- **Residual analysis** reveals model problems\n- **Cross-validation** provides more reliable estimates\n- **Model comparison** should be systematic and data-driven\n- **Interpretation matters** - coefficients tell the story\n\n## Next Steps\n\nIn the next notebook, we'll apply everything to a **Practical Case Study** predicting match outcomes!

## Practice Exercises\n\n1. **Validation Set**: Implement a train/validation/test split (60/20/20)\n2. **Adjusted R²**: Calculate and compare with regular R²\n3. **Prediction Intervals**: Add confidence intervals to predictions\n4. **Feature Selection**: Use p-values to select significant features\n5. **Learning Curves**: Plot training/test scores vs. dataset size